In [ ]:
!pip install openai

In [ ]:
# run this once to clear ERROR rows so they get retried
import csv

for fname in ["EN_Implicature_Results.csv"]:
    rows = []
    with open(fname, encoding="utf-8-sig") as f:
        rows = list(csv.DictReader(f))

    fieldnames = list(rows[0].keys())

    for r in rows:
        if r.get("MODEL_RESPONSE") in ("ERROR", "INVALID"):
            r["MODEL_RESPONSE"] = ""
            r["CORRECT"]        = ""

    with open(fname, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

    cleared = sum(1 for r in rows if r["MODEL_RESPONSE"] == "")
    print(f"Cleared {cleared} ERROR rows in {fname}")

Cleared 150 ERROR rows in EN_Implicature_Results.csv


In [ ]:
import csv

for fname in ["EN_Implicature_Results.csv"]:
    with open(fname, encoding="utf-8-sig") as f:
        rows = list(csv.DictReader(f))
    fieldnames = list(rows[0].keys())
    for r in rows:
        if r.get("MODEL_RESPONSE") in ("ERROR", "INVALID"):
            r["MODEL_RESPONSE"] = ""
            r["CORRECT"] = ""
    with open(fname, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    print(f"Cleared error rows in {fname}")

Cleared error rows in EN_Implicature_Results.csv


In [ ]:
"""
=============================================================
Implicature Testing Script — Single Run
Thesis: Pragmatic Competence of ChatGPT (EN vs BN)
Model: GPT-5.4 mini
=============================================================
HOW TO USE:
1. pip install openai
2. Paste your API key below
3. Put both CSV files in the same folder as this script
=============================================================
"""

import csv, time, re, os
from collections import defaultdict
from openai import OpenAI

# ── SETTINGS ──────────────────────────────────────────────────
API_KEY    = ""   # <-- paste your key
MODEL      = "gpt-5.4-mini"               # GPT-5.4 mini
DELAY      = 0.5                          # seconds between requests
MAX_TOKENS = 20                            # forces single-letter reply
TEMP       = 0                            # deterministic

EN_FILE = "EN_Implicature_Dataset.csv"
BN_FILE = "BN_Implicature_Dataset.csv"
EN_OUT  = "EN_Implicature_Results.csv"
BN_OUT  = "BN_Implicature_Results.csv"
# ──────────────────────────────────────────────────────────────

client = OpenAI(api_key=API_KEY)


def ask(prompt: str) -> str:
    """Send one prompt. Returns 'A', 'B', or 'INVALID'."""
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMP,
            max_completion_tokens=MAX_TOKENS,
        )
        text = resp.choices[0].message.content.strip().upper()
        m = re.search(r'\b([AB])\b', text)
        return m.group(1) if m else "INVALID"
    except Exception as e:
        print(f"\n  API error: {e}")
        return "ERROR"


def load(path: str):
    with open(path, encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))
    return [{k.lstrip('\ufeff'): v for k, v in r.items()} for r in rows]


def save(rows, path: str, fieldnames):
    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)


def run_test(in_file: str, out_file: str):
    print(f"\n{'='*58}")
    print(f"Testing : {in_file}")
    print(f"Output  : {out_file}")
    print(f"{'='*58}")

    rows = load(in_file)
    fieldnames = list(rows[0].keys())
    total = len(rows)

    # resume: skip rows that already have a response
    already = sum(1 for r in rows if r.get("MODEL_RESPONSE","").strip()
                  not in ("", "MODEL_RESPONSE"))
    if already:
        print(f"  Resuming — {already}/{total} already done.")

    invalid = 0

    for i, row in enumerate(rows):

        # skip completed rows
        if row.get("MODEL_RESPONSE","").strip() not in ("", "MODEL_RESPONSE"):
            continue

        iid      = row.get("ITEM_ID", f"row_{i}")
        prompt   = row.get("PROMPT", "")
        expected = row.get("EXPECTED", "")

        print(f"  [{i+1:>3}/{total}] {iid:<28}", end=" ", flush=True)

        response = ask(prompt)
        time.sleep(DELAY)

        correct = (response == expected)
        if response in ("INVALID", "ERROR"):
            invalid += 1

        row["MODEL_RESPONSE"] = response
        row["CORRECT"]        = str(correct)

        tick = "✓" if correct else "✗"
        print(f"got={response}  expected={expected}  {tick}")

        # save after every item (safe to stop and restart)
        save(rows, out_file, fieldnames)

    done    = [r for r in rows if r.get("MODEL_RESPONSE","").strip()
               not in ("", "MODEL_RESPONSE")]
    correct = sum(1 for r in done if r["CORRECT"] == "True")
    acc     = correct / len(done) * 100 if done else 0

    print(f"\n  Done. {len(done)}/{total} tested.")
    print(f"  Correct : {correct}  |  Accuracy : {acc:.1f}%")
    if invalid:
        print(f"  INVALID/ERROR responses: {invalid}")
    print(f"  Saved   : {out_file}")


def breakdown(result_file: str):
    rows  = load(result_file)
    done  = [r for r in rows if r.get("MODEL_RESPONSE","").strip()
             not in ("", "MODEL_RESPONSE")]
    if not done:
        print(f"  No results in {result_file} yet.")
        return

    lang     = done[0].get("LANGUAGE", "?")
    term_nc  = "কিছু" if lang == "BN" else "some"
    term_or  = "অথবা" if lang == "BN" else "or"

    print(f"\n{'='*58}")
    print(f"RESULTS — {lang}  ({result_file})")
    print(f"{'='*58}")
    print(f"  {'TERM':<10}  {'CONDITION':<24}  {'OK':>4}/{'':<4}  {'ACC':>6}")
    print(f"  {'-'*54}")

    groups = defaultdict(list)
    for r in done:
        key = (r.get("SCALAR_TERM",""), r.get("CONDITION",""))
        groups[key].append(r.get("CORRECT") == "True")

    order = [
        (term_nc, "no_context"),
        (term_nc, "upper_bound_QUD"),
        (term_nc, "lower_bound_QUD"),
        (term_or, "no_context"),
        (term_or, "exclusive_context"),
        (term_or, "inclusive_context"),
    ]
    for term, cond in order:
        vals = groups.get((term, cond), [])
        if not vals: continue
        acc = sum(vals) / len(vals) * 100
        print(f"  {term:<10}  {cond:<24}  {sum(vals):>3}/{len(vals):<3}  {acc:>5.1f}%")

    # QUD Sensitivity Gap
    ub = groups.get((term_nc, "upper_bound_QUD"), [])
    lb = groups.get((term_nc, "lower_bound_QUD"), [])
    if ub and lb:
        gap = (sum(ub)/len(ub) - sum(lb)/len(lb)) * 100
        print(f"\n  QUD Sensitivity Gap ({term_nc}): {gap:+.1f}pp")
        print(f"    upper_bound_QUD = {sum(ub)/len(ub)*100:.1f}%")
        print(f"    lower_bound_QUD = {sum(lb)/len(lb)*100:.1f}%")
        if   gap > 20: print(f"    → Model is context-sensitive ✓")
        elif gap > 0 : print(f"    → Weakly context-sensitive")
        else         : print(f"    → Context-insensitive (rigid default)")

    # OR Context Shift
    ex  = groups.get((term_or, "exclusive_context"), [])
    inc = groups.get((term_or, "inclusive_context"), [])
    nc  = groups.get((term_or, "no_context"), [])
    if ex and inc:
        shift = (sum(ex)/len(ex) - sum(inc)/len(inc)) * 100
        print(f"\n  OR Context Shift ({term_or}): {shift:+.1f}pp")
        if nc:
            print(f"    no_context      = {sum(nc)/len(nc)*100:.1f}%  (inclusive default baseline)")
        print(f"    exclusive_ctx   = {sum(ex)/len(ex)*100:.1f}%")
        print(f"    inclusive_ctx   = {sum(inc)/len(inc)*100:.1f}%")
        if   shift > 20: print(f"    → Model shifts reading based on context ✓")
        else           : print(f"    → Model stays on default reading")


def crosslingual_gap():
    if not os.path.exists(EN_OUT) or not os.path.exists(BN_OUT):
        return
    en = load(EN_OUT); bn = load(BN_OUT)
    en_done = [r for r in en if r.get("MODEL_RESPONSE","").strip()
               not in ("", "MODEL_RESPONSE")]
    bn_done = [r for r in bn if r.get("MODEL_RESPONSE","").strip()
               not in ("", "MODEL_RESPONSE")]
    if not en_done or not bn_done:
        return

    ea = sum(1 for r in en_done if r.get("CORRECT")=="True") / len(en_done) * 100
    ba = sum(1 for r in bn_done if r.get("CORRECT")=="True") / len(bn_done) * 100

    print(f"\n{'='*58}")
    print(f"CROSS-LINGUAL GAP — SCALAR IMPLICATURE")
    print(f"{'='*58}")
    print(f"  EN overall: {ea:.1f}%  ({len(en_done)} items)")
    print(f"  BN overall: {ba:.1f}%  ({len(bn_done)} items)")
    print(f"  EN − BN  : {ea-ba:+.1f}pp")

    # per condition
    eg = defaultdict(list); bg = defaultdict(list)
    for r in en_done:
        eg[(r.get("SCALAR_TERM",""), r.get("CONDITION",""))].append(r.get("CORRECT")=="True")
    for r in bn_done:
        bg[(r.get("SCALAR_TERM",""), r.get("CONDITION",""))].append(r.get("CORRECT")=="True")

    print(f"\n  {'TERM':<8}  {'CONDITION':<24}  {'EN':>6}  {'BN':>6}  {'GAP':>6}")
    print(f"  {'-'*58}")
    pairs = [
        ("some","no_context","কিছু"),
        ("some","upper_bound_QUD","কিছু"),
        ("some","lower_bound_QUD","কিছু"),
        ("or","no_context","অথবা"),
        ("or","exclusive_context","অথবা"),
        ("or","inclusive_context","অথবা"),
    ]
    for et, cond, bt in pairs:
        ev = eg.get((et,cond),[])
        bv = bg.get((bt,cond),[])
        if ev and bv:
            ea2 = sum(ev)/len(ev)*100; ba2 = sum(bv)/len(bv)*100
            print(f"  {et:<8}  {cond:<24}  {ea2:>5.1f}%  {ba2:>5.1f}%  {ea2-ba2:>+5.1f}pp")


# ── MAIN ──────────────────────────────────────────────────────
if __name__ == "__main__":
    print("IMPLICATURE TEST — GPT-5.4 mini (single run per item)")
    print(f"Model: {MODEL}  |  Delay: {DELAY}s/request")
    print(f"Total API calls: 300 (150 EN + 150 BN)")
    print(f"Estimated time: ~{300 * DELAY / 60:.1f} minutes\n")

    for f in [EN_FILE, BN_FILE]:
        if not os.path.exists(f):
            print(f"ERROR: {f} not found. Put it in the same folder as this script.")
            exit(1)

    run_test(EN_FILE, EN_OUT)
    run_test(BN_FILE, BN_OUT)
    breakdown(EN_OUT)
    breakdown(BN_OUT)
    crosslingual_gap()

    print(f"\nAll done. Results saved to:\n  {EN_OUT}\n  {BN_OUT}")

IMPLICATURE TEST — GPT-5.4 mini (single run per item)
Model: gpt-5.4-mini  |  Delay: 0.5s/request
Total API calls: 300 (150 EN + 150 BN)
Estimated time: ~2.5 minutes


Testing : EN_Implicature_Dataset.csv
Output  : EN_Implicature_Results.csv
  [  1/150] EN-SOME-001-NC               got=B  expected=A  ✗
  [  2/150] EN-SOME-001-UB               got=A  expected=B  ✗
  [  3/150] EN-SOME-001-LB               got=B  expected=B  ✓
  [  4/150] EN-SOME-002-NC               got=A  expected=B  ✗
  [  5/150] EN-SOME-002-UB               got=B  expected=A  ✗
  [  6/150] EN-SOME-002-LB               got=A  expected=A  ✓
  [  7/150] EN-SOME-003-NC               got=B  expected=A  ✗
  [  8/150] EN-SOME-003-UB               got=A  expected=B  ✗
  [  9/150] EN-SOME-003-LB               got=B  expected=B  ✓
  [ 10/150] EN-SOME-004-NC               got=A  expected=B  ✗
  [ 11/150] EN-SOME-004-UB               got=B  expected=A  ✗
  [ 12/150] EN-SOME-004-LB               got=A  expected=A  ✓
  [ 13/150] EN

In [ ]:
"""
=============================================================
Presupposition Dataset Testing Script — Single Run
Thesis: Pragmatic Competence of ChatGPT (EN vs BN)
Model: GPT-5.4 mini
=============================================================
HOW TO USE:
1. pip install openai
2. Paste your API key below
3. Put both CSV files in the same folder as this script
=============================================================
Total API calls: 200 (100 EN + 100 BN) — single run only
=============================================================
Note: run the same libraries again since it take too long to test one part so re run the libraries to avoid errors
"""

import csv, time, re, os
from collections import defaultdict
from openai import OpenAI

# ── SETTINGS ──────────────────────────────────────────────────
API_KEY    = ""   # <-- paste your key
MODEL      = "gpt-5.4-mini"               # GPT-5.4 mini
DELAY      = 0.5                          # seconds between requests
MAX_TOKENS = 20                            # forces single-letter reply
TEMP       = 0                            # deterministic

EN_FILE = "EN_Presupposition_Dataset.csv"
BN_FILE = "BN_Presupposition_Dataset.csv"
EN_OUT  = "EN_Presupposition_Results.csv"
BN_OUT  = "BN_Presupposition_Results.csv"
# ──────────────────────────────────────────────────────────────

client = OpenAI(api_key=API_KEY)


def ask(prompt: str) -> str:
    """Send one prompt. Returns 'A', 'B', or 'INVALID'."""
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMP,
            max_completion_tokens=MAX_TOKENS,
        )
        text = resp.choices[0].message.content.strip().upper()
        m = re.search(r'\b([AB])\b', text)
        return m.group(1) if m else "INVALID"
    except Exception as e:
        print(f"\n  API error: {e}")
        return "ERROR"


def load(path: str):
    with open(path, encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))
    return [{k.lstrip('\ufeff'): v for k, v in r.items()} for r in rows]


def save(rows, path: str, fieldnames):
    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)


def run_test(in_file: str, out_file: str):
    print(f"\n{'='*58}")
    print(f"Testing : {in_file}")
    print(f"Output  : {out_file}")
    print(f"{'='*58}")

    rows = load(in_file)
    if not rows:
        print("  File is empty.")
        return

    fieldnames = list(rows[0].keys())
    total = len(rows)

    # resume: skip rows already tested
    already = sum(1 for r in rows
                  if r.get("MODEL_RESPONSE", "").strip()
                  not in ("", "MODEL_RESPONSE"))
    if already:
        print(f"  Resuming — {already}/{total} already done.")

    invalid = 0

    for i, row in enumerate(rows):

        # skip completed rows
        if row.get("MODEL_RESPONSE", "").strip() not in ("", "MODEL_RESPONSE"):
            continue

        iid      = row.get("ITEM_ID", f"row_{i}")
        prompt   = row.get("PROMPT", "")
        expected = row.get("EXPECTED", "")

        print(f"  [{i+1:>3}/{total}] {iid:<30}", end=" ", flush=True)

        response = ask(prompt)
        time.sleep(DELAY)

        correct = (response == expected)
        if response in ("INVALID", "ERROR"):
            invalid += 1

        row["MODEL_RESPONSE"] = response
        row["CORRECT"]        = str(correct)

        tick = "✓" if correct else "✗"
        print(f"got={response}  expected={expected}  {tick}")

        # save after every item — safe to stop and restart
        save(rows, out_file, fieldnames)

    done    = [r for r in rows
               if r.get("MODEL_RESPONSE", "").strip()
               not in ("", "MODEL_RESPONSE")]
    correct = sum(1 for r in done if r.get("CORRECT") == "True")
    acc     = correct / len(done) * 100 if done else 0

    print(f"\n  Done. {len(done)}/{total} tested.")
    print(f"  Correct : {correct}  |  Accuracy : {acc:.1f}%")
    if invalid:
        print(f"  INVALID/ERROR responses: {invalid}")
    print(f"  Saved   : {out_file}")


def breakdown(result_file: str):
    """Print accuracy by trigger type and condition."""
    rows = load(result_file)
    done = [r for r in rows
            if r.get("MODEL_RESPONSE", "").strip()
            not in ("", "MODEL_RESPONSE")]
    if not done:
        print(f"  No results in {result_file} yet.")
        return

    lang = done[0].get("LANGUAGE", "?")
    print(f"\n{'='*58}")
    print(f"RESULTS — {lang}  ({result_file})")
    print(f"{'='*58}")

    # overall
    total   = len(done)
    correct = sum(1 for r in done if r.get("CORRECT") == "True")
    print(f"  Overall: {correct}/{total}  ({correct/total*100:.1f}%)")

    # by condition
    groups = defaultdict(list)
    for r in done:
        ttype = r.get("TRIGGER_TYPE", "")
        cond  = r.get("CONDITION", "")
        groups[(ttype, cond)].append(r.get("CORRECT") == "True")

    print(f"\n  {'TRIGGER':<22}  {'CONDITION':<14}  {'OK':>4}/{'':<4}  {'ACC':>6}")
    print(f"  {'-'*54}")

    order = [
        ("change_of_state",   "affirmative"),
        ("change_of_state",   "negated"),
        ("possessed_definite","affirmative"),
        ("possessed_definite","negated"),
    ]
    for ttype, cond in order:
        vals = groups.get((ttype, cond), [])
        if not vals:
            continue
        acc = sum(vals) / len(vals) * 100
        print(f"  {ttype:<22}  {cond:<14}  "
              f"{sum(vals):>3}/{len(vals):<4}  {acc:>5.1f}%")

    # Negation Consistency Ratio — the KEY metric
    print(f"\n  KEY METRIC — Negation Consistency Ratio")
    print(f"  (neg accuracy / aff accuracy — close to 1.0 = true presupposition)")
    print(f"  {'-'*54}")
    for ttype in ["change_of_state", "possessed_definite"]:
        aff_vals = groups.get((ttype, "affirmative"), [])
        neg_vals = groups.get((ttype, "negated"), [])
        if aff_vals and neg_vals:
            aff_acc = sum(aff_vals) / len(aff_vals)
            neg_acc = sum(neg_vals) / len(neg_vals)
            ratio   = neg_acc / aff_acc if aff_acc > 0 else 0
            print(f"\n  {ttype}:")
            print(f"    Affirmative accuracy : {aff_acc*100:.1f}%")
            print(f"    Negated accuracy     : {neg_acc*100:.1f}%")
            print(f"    Negation Consistency : {ratio:.2f}")
            if ratio >= 0.90:
                print(f"    → Genuine presupposition projection ✓")
            elif ratio >= 0.70:
                print(f"    → Partial projection (some entailment confusion)")
            else:
                print(f"    → Entailment confusion — model is fooled by negation ✗")

    # per-sentence diagnostic: which sentences failed in NEG but passed in AFF
    print(f"\n  SENTENCE-LEVEL DIAGNOSIS")
    print(f"  (passed AFF but failed NEG = entailment confusion on that sentence)")
    print(f"  {'-'*54}")
    by_sid = defaultdict(dict)
    for r in done:
        sid  = r.get("SENTENCE_ID", "")
        cond = r.get("CONDITION", "")
        by_sid[sid][cond] = r.get("CORRECT") == "True"

    confused = [(sid, d) for sid, d in by_sid.items()
                if d.get("affirmative") is True and d.get("negated") is False]
    both_correct = [(sid, d) for sid, d in by_sid.items()
                    if d.get("affirmative") is True and d.get("negated") is True]
    both_wrong   = [(sid, d) for sid, d in by_sid.items()
                    if d.get("affirmative") is False and d.get("negated") is False]

    print(f"  Both conditions correct : {len(both_correct)}")
    print(f"  Entailment confusion    : {len(confused)}  "
          f"(passed AFF, failed NEG)")
    print(f"  Both conditions wrong   : {len(both_wrong)}  "
          f"(general comprehension failure)")

    if confused:
        print(f"\n  Confused sentences (first 5):")
        for sid, _ in confused[:5]:
            # find the sentences
            aff_row = next((r for r in done
                            if r.get("SENTENCE_ID")==sid
                            and r.get("CONDITION")=="affirmative"), None)
            neg_row = next((r for r in done
                            if r.get("SENTENCE_ID")==sid
                            and r.get("CONDITION")=="negated"), None)
            if aff_row and neg_row:
                print(f"    [{sid}]")
                print(f"      AFF: {aff_row.get('SENTENCE','')}")
                print(f"      NEG: {neg_row.get('SENTENCE','')}")
                print(f"      Presupposition: {aff_row.get('PRESUPPOSITION','')}")


def crosslingual_gap():
    """Compare EN vs BN results."""
    if not os.path.exists(EN_OUT) or not os.path.exists(BN_OUT):
        return

    en = load(EN_OUT)
    bn = load(BN_OUT)

    en_done = [r for r in en if r.get("MODEL_RESPONSE","").strip()
               not in ("","MODEL_RESPONSE")]
    bn_done = [r for r in bn if r.get("MODEL_RESPONSE","").strip()
               not in ("","MODEL_RESPONSE")]
    if not en_done or not bn_done:
        return

    print(f"\n{'='*58}")
    print(f"CROSS-LINGUAL GAP — PRESUPPOSITION")
    print(f"{'='*58}")

    def acc(rows):
        return sum(1 for r in rows if r.get("CORRECT")=="True") / len(rows) * 100

    en_acc = acc(en_done)
    bn_acc = acc(bn_done)
    print(f"  EN overall : {en_acc:.1f}%  ({len(en_done)} items)")
    print(f"  BN overall : {bn_acc:.1f}%  ({len(bn_done)} items)")
    print(f"  EN − BN    : {en_acc-bn_acc:+.1f}pp")

    # per trigger × condition
    def group(rows):
        g = defaultdict(list)
        for r in rows:
            g[(r.get("TRIGGER_TYPE",""), r.get("CONDITION",""))].append(
                r.get("CORRECT")=="True")
        return g

    eg = group(en_done)
    bg = group(bn_done)

    print(f"\n  {'TRIGGER':<22}  {'CONDITION':<14}  "
          f"{'EN':>6}  {'BN':>6}  {'GAP':>7}")
    print(f"  {'-'*62}")

    for ttype, cond in [("change_of_state","affirmative"),
                        ("change_of_state","negated"),
                        ("possessed_definite","affirmative"),
                        ("possessed_definite","negated")]:
        ev = eg.get((ttype,cond),[])
        bv = bg.get((ttype,cond),[])
        if ev and bv:
            ea = sum(ev)/len(ev)*100
            ba = sum(bv)/len(bv)*100
            print(f"  {ttype:<22}  {cond:<14}  "
                  f"{ea:>5.1f}%  {ba:>5.1f}%  {ea-ba:>+6.1f}pp")

    # Negation Consistency Ratio comparison
    print(f"\n  NEGATION CONSISTENCY RATIO — EN vs BN")
    print(f"  {'-'*50}")
    for ttype in ["change_of_state","possessed_definite"]:
        for lang, gdict, label in [(en_done, eg, "EN"), (bn_done, bg, "BN")]:
            aff = gdict.get((ttype,"affirmative"),[])
            neg = gdict.get((ttype,"negated"),[])
            if aff and neg:
                r = (sum(neg)/len(neg)) / (sum(aff)/len(aff)) \
                    if sum(aff) > 0 else 0
                print(f"  {label}  {ttype:<22}: ratio = {r:.2f}  "
                      f"(aff={sum(aff)/len(aff)*100:.1f}%  "
                      f"neg={sum(neg)/len(neg)*100:.1f}%)")


# ── MAIN ──────────────────────────────────────────────────────
if __name__ == "__main__":
    print("PRESUPPOSITION TEST — GPT-5.4 mini (single run per item)")
    print(f"Model: {MODEL}  |  Delay: {DELAY}s/request")
    print(f"Total API calls: 200 (100 EN + 100 BN)")
    print(f"Estimated time : ~{200 * DELAY / 60:.1f} minutes\n")

    for f in [EN_FILE, BN_FILE]:
        if not os.path.exists(f):
            print(f"ERROR: {f} not found. Put it in the same folder as this script.")
            exit(1)

    run_test(EN_FILE, EN_OUT)
    run_test(BN_FILE, BN_OUT)

    breakdown(EN_OUT)
    breakdown(BN_OUT)
    crosslingual_gap()

    print(f"\nAll done. Results saved to:\n  {EN_OUT}\n  {BN_OUT}")

PRESUPPOSITION TEST — GPT-5.4 mini (single run per item)
Model: gpt-5.4-mini  |  Delay: 0.5s/request
Total API calls: 200 (100 EN + 100 BN)
Estimated time : ~1.7 minutes


Testing : EN_Presupposition_Dataset.csv
Output  : EN_Presupposition_Results.csv
  [  1/100] EN-COS-001-AFF                 got=A  expected=A  ✓
  [  2/100] EN-COS-001-NEG                 got=B  expected=B  ✓
  [  3/100] EN-COS-002-AFF                 got=A  expected=A  ✓
  [  4/100] EN-COS-002-NEG                 got=B  expected=B  ✓
  [  5/100] EN-COS-003-AFF                 got=A  expected=A  ✓
  [  6/100] EN-COS-003-NEG                 got=B  expected=B  ✓
  [  7/100] EN-COS-004-AFF                 got=A  expected=A  ✓
  [  8/100] EN-COS-004-NEG                 got=B  expected=B  ✓
  [  9/100] EN-COS-005-AFF                 got=A  expected=A  ✓
  [ 10/100] EN-COS-005-NEG                 got=B  expected=B  ✓
  [ 11/100] EN-COS-006-AFF                 got=A  expected=A  ✓
  [ 12/100] EN-COS-006-NEG                 g

In [ ]:
"""
=============================================================
Irony / Sarcasm Dataset Testing Script — Single Run
Thesis: Pragmatic Competence of ChatGPT (EN vs BN)
Model: GPT-5.4 mini
=============================================================
HOW TO USE:
1. pip install openai
2. Paste your API key below
3. Put both CSV files in the same folder as this script
=============================================================
Total API calls: 100 (50 EN + 50 BN) — single run only
=============================================================
Note: run the same libraries again since it take too long to test one part so re run the libraries to avoid errors
"""

import csv, time, re, os
from collections import defaultdict
from openai import OpenAI

# ── SETTINGS ──────────────────────────────────────────────────
API_KEY    = ""   # <-- paste your key
MODEL      = "gpt-5.4-mini"               # GPT-5.4 mini
DELAY      = 0.5                          # seconds between requests
MAX_TOKENS = 20                            # forces single-letter reply
TEMP       = 0                            # deterministic

EN_FILE = "EN_Irony_Dataset.csv"
BN_FILE = "BN_Irony_Dataset.csv"
EN_OUT  = "EN_Irony_Results.csv"
BN_OUT  = "BN_Irony_Results.csv"
# ──────────────────────────────────────────────────────────────

client = OpenAI(api_key=API_KEY)


def ask(prompt: str) -> str:
    """Send one prompt. Returns 'A', 'B', or 'INVALID'."""
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMP,
            max_completion_tokens=MAX_TOKENS,
        )
        text = resp.choices[0].message.content.strip().upper()
        m = re.search(r'\b([AB])\b', text)
        return m.group(1) if m else "INVALID"
    except Exception as e:
        print(f"\n  API error: {e}")
        return "ERROR"


def load(path: str):
    with open(path, encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))
    return [{k.lstrip('\ufeff'): v for k, v in r.items()} for r in rows]


def save(rows, path: str, fieldnames):
    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)


def run_test(in_file: str, out_file: str):
    print(f"\n{'='*58}")
    print(f"Testing : {in_file}")
    print(f"Output  : {out_file}")
    print(f"{'='*58}")

    rows = load(in_file)
    if not rows:
        print("  File is empty.")
        return

    fieldnames = list(rows[0].keys())
    total      = len(rows)

    # resume: skip rows already tested
    already = sum(1 for r in rows
                  if r.get("MODEL_RESPONSE", "").strip()
                  not in ("", "MODEL_RESPONSE"))
    if already:
        print(f"  Resuming — {already}/{total} already done.")

    invalid = 0

    for i, row in enumerate(rows):

        # skip completed rows
        if row.get("MODEL_RESPONSE", "").strip() not in ("", "MODEL_RESPONSE"):
            continue

        iid      = row.get("ITEM_ID", f"row_{i}")
        prompt   = row.get("PROMPT", "")
        expected = row.get("EXPECTED", "")

        print(f"  [{i+1:>2}/{total}] {iid:<20}", end=" ", flush=True)

        response = ask(prompt)
        time.sleep(DELAY)

        correct = (response == expected)
        if response in ("INVALID", "ERROR"):
            invalid += 1

        row["MODEL_RESPONSE"] = response
        row["CORRECT"]        = str(correct)

        tick = "✓" if correct else "✗"
        print(f"got={response}  expected={expected}  [{row['LABEL']}]  {tick}")

        # save after every item — safe to stop and restart
        save(rows, out_file, fieldnames)

    done    = [r for r in rows
               if r.get("MODEL_RESPONSE", "").strip()
               not in ("", "MODEL_RESPONSE")]
    correct = sum(1 for r in done if r.get("CORRECT") == "True")
    acc     = correct / len(done) * 100 if done else 0

    print(f"\n  Done. {len(done)}/{total} tested.")
    print(f"  Correct : {correct}  |  Accuracy : {acc:.1f}%")
    if invalid:
        print(f"  INVALID/ERROR responses: {invalid}")
    print(f"  Saved   : {out_file}")


def breakdown(result_file: str):
    """Print accuracy split by sarcastic vs sincere items."""
    rows = load(result_file)
    done = [r for r in rows
            if r.get("MODEL_RESPONSE", "").strip()
            not in ("", "MODEL_RESPONSE")]
    if not done:
        print(f"  No results in {result_file} yet.")
        return

    lang = done[0].get("LANGUAGE", "?")
    print(f"\n{'='*58}")
    print(f"RESULTS — {lang}  ({result_file})")
    print(f"{'='*58}")

    sarc_rows = [r for r in done if r.get("LABEL") == "sarcastic"]
    sinc_rows = [r for r in done if r.get("LABEL") == "sincere"]

    sarc_correct = sum(1 for r in sarc_rows if r.get("CORRECT") == "True")
    sinc_correct = sum(1 for r in sinc_rows if r.get("CORRECT") == "True")

    total   = len(done)
    overall = sum(1 for r in done if r.get("CORRECT") == "True")

    print(f"  Overall accuracy   : {overall}/{total}  "
          f"({overall/total*100:.1f}%)")
    print(f"\n  {'LABEL':<12}  {'CORRECT':>7}  {'TOTAL':>5}  {'ACC':>6}")
    print(f"  {'-'*38}")
    print(f"  {'sarcastic':<12}  {sarc_correct:>7}  {len(sarc_rows):>5}  "
          f"{sarc_correct/len(sarc_rows)*100:>5.1f}%")
    print(f"  {'sincere':<12}  {sinc_correct:>7}  {len(sinc_rows):>5}  "
          f"{sinc_correct/len(sinc_rows)*100:>5.1f}%")

    # KEY METRICS
    print(f"\n  KEY METRICS")
    print(f"  {'-'*50}")

    det_acc  = sarc_correct / len(sarc_rows) * 100  if sarc_rows else 0
    spec_acc = sinc_correct / len(sinc_rows) * 100  if sinc_rows else 0

    print(f"  Irony detection accuracy : {det_acc:.1f}%")
    print(f"    (% of sarcastic items correctly identified)")
    if det_acc >= 70:
        print(f"    → Model detects sarcasm markers well ✓")
    elif det_acc >= 50:
        print(f"    → Moderate sarcasm detection")
    else:
        print(f"    → Model struggles to detect sarcasm ✗")

    print(f"\n  Non-sarcasm specificity  : {spec_acc:.1f}%")
    print(f"    (% of sincere items correctly identified as sincere)")
    if spec_acc >= 70:
        print(f"    → Model avoids false sarcasm detection ✓")
    else:
        print(f"    → Model over-detects sarcasm (false positives) ✗")

    # Bias check — does model default to one label?
    model_sarc = sum(1 for r in done
                     if r.get("MODEL_RESPONSE") == r.get("OPTION_A","?")[0]
                     and "sarcastic" in r.get("OPTION_A",""))
    # simpler: count how many times model chose the sarcastic option
    def chose_sarcastic(r):
        resp = r.get("MODEL_RESPONSE","")
        if resp == "A":
            return "sarcastic" in r.get("OPTION_A","").lower()
        elif resp == "B":
            return "sarcastic" in r.get("OPTION_B","").lower()
        return False

    sarc_responses = sum(1 for r in done if chose_sarcastic(r))
    print(f"\n  Response bias check:")
    print(f"    Model chose sarcastic option: {sarc_responses}/{total} "
          f"({sarc_responses/total*100:.1f}%)")
    if sarc_responses > 35:
        print(f"    → Bias toward sarcastic — model over-detects irony")
    elif sarc_responses < 15:
        print(f"    → Bias toward sincere — model under-detects irony")
    else:
        print(f"    → No strong response bias ✓")

    # Failed items — which ones did it get wrong?
    wrong_sarc = [r for r in sarc_rows if r.get("CORRECT") == "False"]
    wrong_sinc = [r for r in sinc_rows if r.get("CORRECT") == "False"]

    if wrong_sarc:
        print(f"\n  Missed sarcasm (first 5 — model chose sincere):")
        for r in wrong_sarc[:5]:
            print(f"    [{r['ITEM_ID']}] {r['TWEET']}")

    if wrong_sinc:
        print(f"\n  False sarcasm detection (first 5 — model chose sarcastic):")
        for r in wrong_sinc[:5]:
            print(f"    [{r['ITEM_ID']}] {r['TWEET']}")


def crosslingual_gap():
    """Compare EN vs BN irony results."""
    if not os.path.exists(EN_OUT) or not os.path.exists(BN_OUT):
        return

    en = load(EN_OUT)
    bn = load(BN_OUT)

    en_done = [r for r in en if r.get("MODEL_RESPONSE","").strip()
               not in ("","MODEL_RESPONSE")]
    bn_done = [r for r in bn if r.get("MODEL_RESPONSE","").strip()
               not in ("","MODEL_RESPONSE")]
    if not en_done or not bn_done:
        return

    print(f"\n{'='*58}")
    print(f"CROSS-LINGUAL GAP — IRONY & SARCASM")
    print(f"{'='*58}")

    def metrics(rows):
        sarc = [r for r in rows if r.get("LABEL")=="sarcastic"]
        sinc = [r for r in rows if r.get("LABEL")=="sincere"]
        overall = sum(1 for r in rows if r.get("CORRECT")=="True") / len(rows) * 100
        det  = sum(1 for r in sarc if r.get("CORRECT")=="True") / len(sarc) * 100 if sarc else 0
        spec = sum(1 for r in sinc if r.get("CORRECT")=="True") / len(sinc) * 100 if sinc else 0
        return overall, det, spec

    en_ov, en_det, en_spec = metrics(en_done)
    bn_ov, bn_det, bn_spec = metrics(bn_done)

    print(f"\n  {'METRIC':<30}  {'EN':>6}  {'BN':>6}  {'GAP':>7}")
    print(f"  {'-'*56}")
    print(f"  {'Overall accuracy':<30}  {en_ov:>5.1f}%  {bn_ov:>5.1f}%  "
          f"{en_ov-bn_ov:>+6.1f}pp")
    print(f"  {'Irony detection accuracy':<30}  {en_det:>5.1f}%  {bn_det:>5.1f}%  "
          f"{en_det-bn_det:>+6.1f}pp")
    print(f"  {'Non-sarcasm specificity':<30}  {en_spec:>5.1f}%  {bn_spec:>5.1f}%  "
          f"{en_spec-bn_spec:>+6.1f}pp")

    gap = en_ov - bn_ov
    if gap > 15:
        print(f"\n  → Large English advantage ({gap:+.1f}pp) — sarcasm markers "
              f"better encoded in English training data")
    elif gap > 5:
        print(f"\n  → Moderate English advantage ({gap:+.1f}pp)")
    else:
        print(f"\n  → Minimal cross-lingual gap for irony ({gap:+.1f}pp)")


# ── MAIN ──────────────────────────────────────────────────────
if __name__ == "__main__":
    print("IRONY / SARCASM TEST — GPT-5.4 mini (single run per item)")
    print(f"Model: {MODEL}  |  Delay: {DELAY}s/request")
    print(f"Total API calls: 100 (50 EN + 50 BN)")
    print(f"Estimated time : ~{100 * DELAY / 60:.1f} minutes\n")

    for f in [EN_FILE, BN_FILE]:
        if not os.path.exists(f):
            print(f"ERROR: {f} not found. "
                  f"Put it in the same folder as this script.")
            exit(1)

    run_test(EN_FILE, EN_OUT)
    run_test(BN_FILE, BN_OUT)

    breakdown(EN_OUT)
    breakdown(BN_OUT)
    crosslingual_gap()

    print(f"\nAll done. Results saved to:\n  {EN_OUT}\n  {BN_OUT}")

IRONY / SARCASM TEST — GPT-5.4 mini (single run per item)
Model: gpt-5.4-mini  |  Delay: 0.5s/request
Total API calls: 100 (50 EN + 50 BN)
Estimated time : ~0.8 minutes


Testing : EN_Irony_Dataset.csv
Output  : EN_Irony_Results.csv
  [ 1/50] EN-IRO-001           got=B  expected=B  [sincere]  ✓
  [ 2/50] EN-IRO-002           got=B  expected=B  [sarcastic]  ✓
  [ 3/50] EN-IRO-003           got=A  expected=A  [sarcastic]  ✓
  [ 4/50] EN-IRO-004           got=B  expected=B  [sarcastic]  ✓
  [ 5/50] EN-IRO-005           got=A  expected=A  [sarcastic]  ✓
  [ 6/50] EN-IRO-006           got=A  expected=A  [sincere]  ✓
  [ 7/50] EN-IRO-007           got=B  expected=B  [sincere]  ✓
  [ 8/50] EN-IRO-008           got=B  expected=B  [sarcastic]  ✓
  [ 9/50] EN-IRO-009           got=B  expected=B  [sincere]  ✓
  [10/50] EN-IRO-010           got=B  expected=B  [sarcastic]  ✓
  [11/50] EN-IRO-011           got=B  expected=B  [sincere]  ✓
  [12/50] EN-IRO-012           got=B  expected=B  [sarcastic] 